In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/omr-data-preprocessing


/home/dongmin/userdata/dongmin/omr-data-preprocessing/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index
from modules.lmxe.lmxe.Linearizer import LMXELinearizer
from modules.lmxe.lmxe.Delinearizer import LMXEDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score
from modules.lmxe.lmxe.symbolic.debug_compare import compare_parts

from omrdp.utils import load_ossq_metadata
from omrdp.utils import parse_log_and_get_exclusions, print_exclusion_report
from omrdp.utils.image import crop_white_space

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

## Load metadata

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')
len(metadata)

122

In [5]:
temp_dir = Path.cwd() / 'temp'
temp_dir.mkdir(exist_ok=True)

## MusicXML to abc

In [6]:
fatal_error_set = {
  'conversion error',
  'measure insert/delete', 
  'staff insert/delete',
  'staff group',
  'barline',
  'keysig', 'clef', 'tie',
  'note', 'dot',
}

### Synthetic

In [7]:
abc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/musicxml_to_abc_synthetic_2026-04-03-15:45:17.log'

eabc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/abc_to_eabc_synthetic_systemwise_2026-04-03-16:25:18.log'

excluded_segments_path = dataset_dir / 'excluded_segments_synthetic_abc_systemwise.txt'

In [8]:
abc_log_file = Path(abc_log_file)
(
  abc_excluded_segments, 
  abc_num_samples, 
  abc_known_errors, 
  abc_known_fatal_errors, 
  abc_error_cnt, 
  abc_failed_cnt
) = parse_log_and_get_exclusions(
  abc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"ABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=abc_num_samples,
  excluded_segments=abc_excluded_segments,
  known_errors=abc_known_errors,
  known_fatal_errors=abc_known_fatal_errors,
  error_cnt=abc_error_cnt,
  failed_cnt=abc_failed_cnt,
)

ABC Conversion Exclusion Report:
total: 13240
excluded segments: 2805
exclusion rate: 21.185800604229605

error segments: 2805
failed segments: 0

Known fatal errors:
{
  "conversion error": 4,
  "tie": 3370,
  "clef": 1276,
  "note": 24765,
  "measure insert/delete": 4616,
  "keysig": 272,
  "dot": 1029
}

Known errors:
{
  "conversion error": 4,
  "tie": 3370,
  "slur": 39423,
  "accidental": 1009,
  "clef": 1276,
  "note": 24765,
  "measure insert/delete": 4616,
  "keysig": 272,
  "flag/beam": 947,
  "note head": 406,
  "dot": 1029,
  "fingered tremolo": 297,
  "ornament": 336,
  "multi-staff arpeggio": 126,
  "articulation": 1
}


In [9]:
eabc_log_file = Path(eabc_log_file)
(
  eabc_excluded_segments, 
  eabc_num_samples, 
  eabc_known_errors, 
  eabc_known_fatal_errors, 
  eabc_error_cnt, 
  eabc_failed_cnt
) = parse_log_and_get_exclusions(
  eabc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=eabc_num_samples,
  excluded_segments=eabc_excluded_segments,
  known_errors=eabc_known_errors,
  known_fatal_errors=eabc_known_fatal_errors,
  error_cnt=eabc_error_cnt,
  failed_cnt=eabc_failed_cnt,
)

EABC Conversion Exclusion Report:
total: 13241
excluded segments: 1
exclusion rate: 0.007552299675251114

error segments: 0
failed segments: 1

Known fatal errors:
{}

Known errors:
{}


In [10]:
abc_excluded_set = set(abc_excluded_segments)
eabc_excluded_set = set(eabc_excluded_segments)

print("len kern:", len(abc_excluded_set))
print("len ekern:", len(eabc_excluded_set))
print("len intersection:", len(abc_excluded_set & eabc_excluded_set))
print("len union:", len(abc_excluded_set | eabc_excluded_set))
print("len abc - eabc:", len(abc_excluded_set - eabc_excluded_set))
print("len eabc - abc:", len(eabc_excluded_set - abc_excluded_set))

len kern: 2805
len ekern: 1
len intersection: 0
len union: 2806
len abc - eabc: 2805
len eabc - abc: 1


In [11]:
excluded_list = list(abc_excluded_set | eabc_excluded_set)
print(f"Total Segments: {abc_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / abc_num_samples * 100:.2f}%")

Total Segments: 13240
Total Excluded Segments: 2806
Total Exclusion Rate: 21.19%


In [12]:
with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')

### Scanned

In [13]:
abc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/musicxml_to_abc_scanned_2026-04-03-16:00:53.log'

eabc_log_file = '/home/dongmin/userdata/open-score-string-quartets/logs/abc_to_eabc_scanned_systemwise_2026-04-03-16:29:14.log'

excluded_segments_path = dataset_dir / 'excluded_segments_scanned_abc_systemwise.txt'

In [14]:
abc_log_file = Path(abc_log_file)
(
  abc_excluded_segments, 
  abc_num_samples, 
  abc_known_errors, 
  abc_known_fatal_errors, 
  abc_error_cnt, 
  abc_failed_cnt
) = parse_log_and_get_exclusions(
  abc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"ABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=abc_num_samples,
  excluded_segments=abc_excluded_segments,
  known_errors=abc_known_errors,
  known_fatal_errors=abc_known_fatal_errors,
  error_cnt=abc_error_cnt,
  failed_cnt=abc_failed_cnt,
)

ABC Conversion Exclusion Report:
total: 11307
excluded segments: 2456
exclusion rate: 21.721057751835147

error segments: 2456
failed segments: 0

Known fatal errors:
{
  "conversion error": 4,
  "tie": 2980,
  "clef": 1101,
  "measure insert/delete": 5808,
  "note": 24454,
  "keysig": 276,
  "dot": 1011
}

Known errors:
{
  "conversion error": 4,
  "slur": 33763,
  "accidental": 850,
  "tie": 2980,
  "clef": 1101,
  "measure insert/delete": 5808,
  "note": 24454,
  "keysig": 276,
  "flag/beam": 919,
  "note head": 362,
  "dot": 1011,
  "fingered tremolo": 279,
  "ornament": 262,
  "multi-staff arpeggio": 86,
  "articulation": 2
}


In [15]:
eabc_log_file = Path(eabc_log_file)
(
  eabc_excluded_segments, 
  eabc_num_samples, 
  eabc_known_errors, 
  eabc_known_fatal_errors, 
  eabc_error_cnt, 
  eabc_failed_cnt
) = parse_log_and_get_exclusions(
  eabc_log_file,
  metadata,
  dataset_score_dir,
  fatal_error_set=fatal_error_set,
)

print(f"EABC Conversion Exclusion Report:")
print_exclusion_report(
  num_samples=eabc_num_samples,
  excluded_segments=eabc_excluded_segments,
  known_errors=eabc_known_errors,
  known_fatal_errors=eabc_known_fatal_errors,
  error_cnt=eabc_error_cnt,
  failed_cnt=eabc_failed_cnt,
)

EABC Conversion Exclusion Report:
total: 11305
excluded segments: 1
exclusion rate: 0.008845643520566122

error segments: 0
failed segments: 1

Known fatal errors:
{}

Known errors:
{}


In [16]:
abc_excluded_set = set(abc_excluded_segments)
eabc_excluded_set = set(eabc_excluded_segments)

print("len kern:", len(abc_excluded_set))
print("len ekern:", len(eabc_excluded_set))
print("len intersection:", len(abc_excluded_set & eabc_excluded_set))
print("len union:", len(abc_excluded_set | eabc_excluded_set))
print("len abc - eabc:", len(abc_excluded_set - eabc_excluded_set))
print("len eabc - abc:", len(eabc_excluded_set - abc_excluded_set))

len kern: 2453
len ekern: 1
len intersection: 0
len union: 2454
len abc - eabc: 2453
len eabc - abc: 1


In [17]:
excluded_list = list(abc_excluded_set | eabc_excluded_set)
print(f"Total Segments: {abc_num_samples}")
print(f"Total Excluded Segments: {len(excluded_list)}")
print(f"Total Exclusion Rate: {len(excluded_list) / abc_num_samples * 100:.2f}%")

Total Segments: 11307
Total Excluded Segments: 2454
Total Exclusion Rate: 21.70%


In [18]:
with open(excluded_segments_path, 'w') as f:
  for seg_id in excluded_list:
    f.write(seg_id + '\n')